[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/advanced-ab-testing/blob/main/assignments/week04_assignment.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 4 Assignment: Stratified Group Selection

## QuickCart — Reducing Noise with Stratified Assignment

QuickCart has observed that experiment results often show **high variance** because user behavior differs dramatically across segments:
- **Mobile vs Desktop** users have 2x difference in conversion rates
- **New vs Returning** customers have very different average order values
- **iOS vs Android** users show different engagement patterns

When users are randomly assigned to groups, these imbalances introduce noise. **Stratified randomization** ensures that each group has the same proportion of each user segment, reducing variance and increasing the experiment's power to detect real effects.

Your task: implement stratified group selection and demonstrate its advantages over simple random sampling.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from itertools import product as iter_product

## Synthetic User Data

We generate a user-level dataset for QuickCart with stratification-relevant features and a revenue metric.

In [ ]:
np.random.seed(42)

n_users = 10_000

# User attributes (stratification dimensions)
platform = np.random.choice(['ios', 'android', 'desktop'], size=n_users, p=[0.35, 0.30, 0.35])
user_type = np.random.choice(['new', 'returning'], size=n_users, p=[0.4, 0.6])
gender = np.random.choice(['male', 'female'], size=n_users, p=[0.55, 0.45])

# Revenue depends on segments (this creates the heterogeneity we want to control for)
base_revenue = 50.0
revenue = base_revenue + np.random.normal(0, 15, n_users)

# Platform effect
revenue += np.where(platform == 'desktop', 20, 0)
revenue += np.where(platform == 'ios', 10, 0)

# User type effect
revenue += np.where(user_type == 'returning', 15, 0)

# Ensure non-negative
revenue = np.maximum(revenue, 0)

df_users = pd.DataFrame({
    'user_id': np.arange(1, n_users + 1),
    'platform': platform,
    'user_type': user_type,
    'gender': gender,
    'revenue': np.round(revenue, 2)
})

print(f"Users: {len(df_users):,}")
print(f"\nPlatform distribution:")
print(df_users['platform'].value_counts(normalize=True).round(3))
print(f"\nUser type distribution:")
print(df_users['user_type'].value_counts(normalize=True).round(3))
print(f"\nRevenue by platform:")
print(df_users.groupby('platform')['revenue'].mean().round(2))
print(f"\nRevenue by user_type:")
print(df_users.groupby('user_type')['revenue'].mean().round(2))
df_users.head()

---

## Task 1: Implement `select_stratified_groups`

### Problem

Implement a function that creates **stratified pilot and control groups**. The function should:

1. Group users by the specified stratification columns
2. If `weights` is `None`, compute **proportional weights** from the data (each stratum's share of total)
3. For each stratum, compute how many users to allocate: `n_stratum = round(weight * group_size)`
4. Randomly sample that many users for the **pilot** group, and separately for the **control** group (no overlap)
5. Return two DataFrames: `(data_pilot, data_control)`

**Important constraints:**
- Pilot and control must not share any users
- Each group should have approximately `group_size` total users
- The distribution across strata should match the weights

### Function signature

```python
def select_stratified_groups(data, strat_columns, group_size, weights=None, seed=None):
    """
    Parameters
    ----------
    data : pd.DataFrame
        Full user dataset.
    strat_columns : list of str
        Columns to stratify on, e.g., ['platform', 'user_type'].
    group_size : int
        Desired size for each group (pilot and control).
    weights : dict or None
        Mapping from stratum tuple to weight.
        e.g., {('ios', 'new'): 0.14, ('ios', 'returning'): 0.21, ...}
        Weights should sum to 1. If None, compute from data.
    seed : int or None
        Random seed for reproducibility.
    
    Returns
    -------
    tuple of (pd.DataFrame, pd.DataFrame)
        (data_pilot, data_control)
    """
```

<details>
<summary>Hint 1: Computing proportional weights (click to expand)</summary>

```python
if weights is None:
    strat_counts = data.groupby(strat_columns).size()
    weights = (strat_counts / len(data)).to_dict()
```

</details>

<details>
<summary>Hint 2: Sampling within each stratum</summary>

For each stratum:
1. Filter the data to that stratum
2. Compute `n = round(weight * group_size)`
3. Shuffle the stratum users
4. Take the first `n` for pilot, the next `n` for control

This ensures no overlap between groups.

</details>

<details>
<summary>Hint 3: Handling the random state</summary>

```python
rng = np.random.default_rng(seed)
# Then use rng.permutation(indices) for shuffling
```

Or use `data.sample(frac=1, random_state=seed)` for shuffling.

</details>

<details>
<summary>Hint 4: Edge case — stratum too small</summary>

If a stratum has fewer users than `2 * n` (what you need for both groups), you can only take `floor(available / 2)` for each group. Handle this gracefully.

</details>

In [ ]:
def select_stratified_groups(data, strat_columns, group_size, weights=None, seed=None):
    """Create stratified pilot and control groups.
    
    Parameters
    ----------
    data : pd.DataFrame
        Full user dataset.
    strat_columns : list of str
        Columns to stratify on.
    group_size : int
        Desired size for each group (pilot and control).
    weights : dict or None
        Mapping from stratum tuple to weight (should sum to ~1).
        If None, compute proportional weights from data.
    seed : int or None
        Random seed for reproducibility.
    
    Returns
    -------
    tuple of (pd.DataFrame, pd.DataFrame)
        (data_pilot, data_control)
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Test Cases for Task 1 ---

# Test with proportional weights (weights=None)
pilot, control = select_stratified_groups(
    df_users,
    strat_columns=['platform', 'user_type'],
    group_size=2000,
    seed=42
)

print(f"Pilot group size: {len(pilot)}")
print(f"Control group size: {len(control)}")
print()

# Test 1: Groups should be approximately the right size
assert abs(len(pilot) - 2000) < 50, f"Pilot size {len(pilot)} too far from 2000"
assert abs(len(control) - 2000) < 50, f"Control size {len(control)} too far from 2000"
print("Size check passed!")

# Test 2: No overlap between groups
overlap = set(pilot['user_id']).intersection(set(control['user_id']))
assert len(overlap) == 0, f"Groups overlap on {len(overlap)} users!"
print("No-overlap check passed!")

# Test 3: Stratification proportions should match the population
pop_props = df_users.groupby(['platform', 'user_type']).size() / len(df_users)
pilot_props = pilot.groupby(['platform', 'user_type']).size() / len(pilot)
control_props = control.groupby(['platform', 'user_type']).size() / len(control)

print("\nProportion comparison (population vs pilot vs control):")
comparison = pd.DataFrame({
    'population': pop_props,
    'pilot': pilot_props,
    'control': control_props
}).round(3)
print(comparison)

# Proportions should be within 2 percentage points
max_diff = (comparison['pilot'] - comparison['population']).abs().max()
assert max_diff < 0.02, f"Pilot proportions deviate by {max_diff:.3f}"
print(f"\nMax proportion deviation: {max_diff:.4f} (< 0.02)")
print("Stratification check passed!")

In [ ]:
# Test 4: Custom weights
custom_weights = {
    ('ios', 'new'): 0.20,
    ('ios', 'returning'): 0.30,
    ('android', 'new'): 0.15,
    ('android', 'returning'): 0.15,
    ('desktop', 'new'): 0.10,
    ('desktop', 'returning'): 0.10
}

pilot_cw, control_cw = select_stratified_groups(
    df_users,
    strat_columns=['platform', 'user_type'],
    group_size=1000,
    weights=custom_weights,
    seed=123
)

# With custom weights, ios+returning should be ~30% of each group
ios_ret_pilot = len(pilot_cw[(pilot_cw['platform'] == 'ios') & (pilot_cw['user_type'] == 'returning')])
ios_ret_frac = ios_ret_pilot / len(pilot_cw)
print(f"iOS+returning fraction in pilot: {ios_ret_frac:.3f} (target: 0.30)")
assert abs(ios_ret_frac - 0.30) < 0.03, f"Custom weight not respected: got {ios_ret_frac:.3f}"
print("Custom weights test passed!")

# Test 5: Reproducibility with seed
pilot_a, _ = select_stratified_groups(df_users, ['platform'], 1000, seed=99)
pilot_b, _ = select_stratified_groups(df_users, ['platform'], 1000, seed=99)
assert set(pilot_a['user_id']) == set(pilot_b['user_id']), "Same seed should give same result"
print("Reproducibility test passed!")

print("\nAll Task 1 tests passed!")

---

## Task 2: Stratified vs Simple Random Sampling — Variance Comparison

### Problem

Demonstrate empirically that stratified sampling reduces the variance of the estimated treatment effect.

Run **1000 simulations** where in each iteration:
1. **Simple Random Sampling (SRS):** randomly select `group_size` users for pilot and another `group_size` for control
2. **Stratified Sampling:** use your `select_stratified_groups` function
3. Compute the **difference in mean revenue** between pilot and control

Since there is no real treatment effect (we're splitting the same population), the true difference is 0. The variance of the estimated difference tells us how noisy our estimate is.

Compare:
- Variance of the difference under SRS
- Variance of the difference under stratified sampling
- The **variance reduction ratio** (stratified variance / SRS variance)

<details>
<summary>Hint: SRS implementation</summary>

```python
shuffled = data.sample(frac=1, random_state=seed)
srs_pilot = shuffled.iloc[:group_size]
srs_control = shuffled.iloc[group_size:2*group_size]
```

</details>

<details>
<summary>Hint: What to expect</summary>

With stratification on platform + user_type (which explain a lot of variance in revenue), you should see the stratified variance be **30-60% lower** than SRS variance.

</details>

In [ ]:
# YOUR CODE HERE
# 1. Run 1000 simulations of SRS and stratified sampling
# 2. In each, compute mean_revenue_pilot - mean_revenue_control
# 3. Compare the variance of these differences
# 4. Plot histograms of the two distributions side by side

n_simulations = 1000
group_size = 2000
strat_cols = ['platform', 'user_type']

---

## Task 3: Covariate Balance Check

### Problem

Show that stratified sampling achieves **better balance** on key covariates than random assignment.

For a single split (seed=42, group_size=2000):
1. Create groups using both SRS and stratified sampling
2. For each method, compute the **standardized mean difference (SMD)** between pilot and control for:
   - Proportion of each platform (ios, android, desktop)
   - Proportion of each user type (new, returning)
   - Mean revenue
3. Display the results in a table

The **SMD** is defined as:
$$\text{SMD} = \frac{|\bar{x}_{pilot} - \bar{x}_{control}|}{s_{pooled}}$$

where $s_{pooled} = \sqrt{(s^2_{pilot} + s^2_{control})/2}$.

A common threshold: SMD < 0.1 indicates good balance.

<details>
<summary>Hint: Computing SMD for a binary variable</summary>

For a proportion (e.g., fraction of iOS users):
```python
p_pilot = (pilot['platform'] == 'ios').mean()
p_control = (control['platform'] == 'ios').mean()
# For proportions, use sqrt(p*(1-p)) as the std
s_pooled = np.sqrt((p_pilot*(1-p_pilot) + p_control*(1-p_control)) / 2)
smd = abs(p_pilot - p_control) / s_pooled
```

</details>

In [ ]:
# YOUR CODE HERE
# 1. Create SRS groups (seed=42, group_size=2000)
# 2. Create stratified groups (same params)
# 3. Compute SMD for key covariates under both methods
# 4. Display comparison table

---

## Bonus: The Curse of Dimensionality in Stratification

### Problem

What happens if you stratify on **too many columns**? Explore this by:

1. Adding more categorical features to the data (e.g., region with 5 levels, age_group with 4 levels)
2. Trying to stratify on all columns simultaneously
3. Observing:
   - How many strata are created?
   - What fraction of strata have fewer than 2 users (making it impossible to assign one to each group)?
   - Does the group size match the target?

Write a brief explanation of when stratification helps vs when it becomes counterproductive.

<details>
<summary>Hint: Generating more features</summary>

```python
df_users['region'] = np.random.choice(['north', 'south', 'east', 'west', 'central'], n_users)
df_users['age_group'] = np.random.choice(['18-25', '26-35', '36-50', '50+'], n_users)
# Now stratify on ['platform', 'user_type', 'gender', 'region', 'age_group']
# That's 3 * 2 * 2 * 5 * 4 = 240 strata for 10,000 users (~42 users per stratum)
```

</details>

In [ ]:
# YOUR CODE HERE (optional)
# 1. Add region and age_group columns
# 2. Try stratifying on all 5 columns
# 3. Analyze strata sizes and empty strata
# 4. Compare with stratifying on just 1-2 columns

**Your analysis:**

*When does stratification help? When does it become counterproductive? What is a practical guideline for choosing stratification variables?*